# Winner + Multi-Sample Dropout

Este notebook reproduz o pipeline vencedor (BERTimbau, 5 folds, Focal Loss, calibracao fixa) e adiciona Multi-Sample Dropout (MSD) na cabeca de classificacao.

Ideia do MSD:
- Substitui a classification head padrao do BERT por uma com N=5 passes de dropout.
- No forward, aplica dropout 5x sobre o pooled output, passa pela classifier linear e faz a media dos logits.
- Training: dropout ativo, backward na loss media sobre o batch, gradientes fluem por todas as N passagens.
- Inference: dropouts continuam ativos (nao chama .eval() nos dropouts), o resto do modelo vai para eval. Isso preserva o efeito regularizador e a media de N logits estocasticos funciona como um mini ensemble interno.

Meta: ganho tipico de +0.2 a +0.5 pp em F1 macro no Kaggle em relacao ao pipeline base.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

MAX_LEN      = 512
BATCH_SIZE   = 8
EPOCHS       = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS      = 5
SEED         = 42
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
NUM_CLASSES  = 7

BEST_TEMP       = 0.958
BEST_THRESHOLDS = [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1]

MSD_N_SAMPLES = 5
MSD_DROPOUT   = 0.5

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# ==========================================================================
# AUTO-DETECTAR PATH DO MODELO NO KAGGLE
# ==========================================================================
import os
from pathlib import Path

def find_model_path():
    candidates = [
        '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
        '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
        '/kaggle/input/bert-large-portuguese-cased',
        '/kaggle/input/bertimbau-ptbr-complete',
        '/kaggle/input/bertimbau-large',
        '/kaggle/input/bertimbau',
        '/kaggle/input/neuralmind-bert-large-portuguese-cased',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'config.json')):
            return c
    # Fallback: varrer /kaggle/input procurando config.json com model BERT grande
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    best = None
    for cfg_path in root.rglob('config.json'):
        try:
            with open(cfg_path) as f:
                cfg = json.load(f)
            # BERT large tipicamente tem hidden_size=1024
            if cfg.get('model_type') == 'bert' and cfg.get('hidden_size', 0) >= 1024:
                # Verifica se tem pesos e tokenizer no mesmo diretorio
                parent = cfg_path.parent
                has_tokenizer = any((parent / f).exists() for f in ['tokenizer.json', 'vocab.txt', 'tokenizer_config.json'])
                has_weights = any(p.suffix in ('.bin', '.safetensors') for p in parent.iterdir())
                if has_tokenizer and has_weights:
                    best = str(parent)
                    break
        except Exception:
            continue
    return best

import json as _json
MODEL_PATH = find_model_path()
assert MODEL_PATH is not None, (
    'Modelo BERTimbau Large nao encontrado em /kaggle/input.\n'
    'Adicione como Input: neuralmind/bert-large-portuguese-cased ou equivalente.\n'
    'Arquivos em /kaggle/input:\n  ' + '\n  '.join(
        str(p) for p in Path('/kaggle/input').rglob('config.json')
    )
)
print('MODEL_PATH detectado:', MODEL_PATH)
print('Arquivos:', sorted(os.listdir(MODEL_PATH)))


In [ ]:
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print('Train:', train_df.shape, 'Test:', test_df.shape)

train_texts  = [build_input_text(t) for t in train_df['report'].fillna('').tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].fillna('').tolist()]
print('Exemplo train_text[0]:')
print(train_texts[0][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class SPRDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_token_type_ids=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc['token_type_ids'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = self.alpha * (1.0 - pt) ** self.gamma * ce
        return loss.mean()

class MSDBertClassifier(nn.Module):
    'BERTimbau + Multi-Sample Dropout head.'
    def __init__(self, model_path, num_classes=NUM_CLASSES, n_samples=MSD_N_SAMPLES, dropout=MSD_DROPOUT):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_path, local_files_only=True)
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.n_samples = n_samples
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(n_samples)])
        self.classifier = nn.Linear(self.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = torch.stack(
            [self.classifier(dp(pooled)) for dp in self.dropouts], dim=0
        ).mean(dim=0)
        return logits

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total = 0.0
    n = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item() * labels.size(0)
        n += labels.size(0)
    return total / max(n, 1)

@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    for m in model.dropouts:
        m.train()
    all_probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        probs = F.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

def build_optimizer_scheduler(model, num_training_steps):
    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = AdamW(params, lr=LR)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * num_training_steps),
        num_training_steps=num_training_steps,
    )
    return optimizer, scheduler

In [ ]:
test_ds = SPRDataset(test_df['text'].values, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

oof_probs  = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
test_probs = np.zeros((len(test_df), NUM_CLASSES), dtype=np.float32)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
loss_fn = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df['text'].values, train_labels)):
    print(f'\n=== Fold {fold + 1}/{N_FOLDS} ===')
    set_seed(SEED + fold)

    tr_ds = SPRDataset(train_df['text'].values[tr_idx], train_labels[tr_idx], tokenizer, MAX_LEN)
    va_ds = SPRDataset(train_df['text'].values[va_idx], train_labels[va_idx], tokenizer, MAX_LEN)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = MSDBertClassifier(MODEL_PATH).to(device)
    num_training_steps = len(tr_loader) * EPOCHS
    optimizer, scheduler = build_optimizer_scheduler(model, num_training_steps)

    best_f1 = -1.0
    best_state = None
    for epoch in range(EPOCHS):
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, loss_fn, device)
        va_p = predict(model, va_loader, device)
        va_pred = np.argmax(va_p, axis=1)
        va_f1 = f1_score(train_labels[va_idx], va_pred, average='macro')
        print(f'fold {fold + 1} epoch {epoch + 1}/{EPOCHS} loss {tr_loss:.4f} val_f1 {va_f1:.5f}')
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    oof_probs[va_idx] = predict(model, va_loader, device)
    test_probs += predict(model, test_loader, device) / N_FOLDS

    del model, optimizer, scheduler, best_state
    torch.cuda.empty_cache()

In [ ]:
def temperature_scale(probs, T):
    eps = 1e-12
    logits = np.log(np.clip(probs, eps, 1.0))
    scaled = logits / T
    scaled = scaled - scaled.max(axis=1, keepdims=True)
    exp = np.exp(scaled)
    return exp / exp.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    thr = np.asarray(thresholds, dtype=np.float32).reshape(1, -1)
    adjusted = probs * thr
    return np.argmax(adjusted, axis=1)

oof_cal = temperature_scale(oof_probs, BEST_TEMP)
oof_preds = apply_thresholds(oof_cal, BEST_THRESHOLDS)
oof_f1 = f1_score(train_labels, oof_preds, average='macro')
print(f'OOF F1 (MSD + fixed calibration): {oof_f1:.5f}')

In [ ]:
test_cal = temperature_scale(test_probs, BEST_TEMP)
test_preds = apply_thresholds(test_cal, BEST_THRESHOLDS)
submission = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('submission shape:', submission.shape)